### Read and combine all 44 CME files

In [1]:
import pandas as pd
from pathlib import Path
from zipfile import ZipFile

# Folder containing 21 CME files
data_path = Path("data/CME_Swap_data")

frames = []

for file in sorted(data_path.iterdir()):

    if file.suffix.lower() == ".csv":
        frames.append(pd.read_csv(file, low_memory=False))

    elif file.suffix.lower() == ".zip":
        with ZipFile(file) as z:
            for name in z.namelist():
                if name.lower().endswith(".csv"):
                    with z.open(name) as f:
                        frames.append(pd.read_csv(f, low_memory=False))

# Combine all transactions
cme_df = pd.concat(frames, ignore_index=True)

print("Files processed:", len(frames))
print("Total transactions:", len(cme_df))
print("Total columns:", len(cme_df.columns))

Files processed: 43
Total transactions: 778
Total columns: 148


In [2]:
### Extract GBP transactions
gbp_df = cme_df[
    cme_df["leg1NotionalCurrency"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("GBP")
].copy()

print("Total GBP transactions:", len(gbp_df))

print("\nAction distribution:")
print(gbp_df["action"].value_counts(dropna=False))

print("\nEvent distribution:")
print(gbp_df["event"].value_counts(dropna=False))

display(gbp_df.head())


Total GBP transactions: 102

Action distribution:
action
NEWT    102
Name: count, dtype: int64

Event distribution:
event
TRAD    102
Name: count, dtype: int64


,action,amendmentIndicator,assetClass,blockTradeIndicator,cleared,coupon,customBasketIndicator,deliveryType,disseminationIdentifier,disseminationTimestamp,...,leg1QuantityFrequencyPeriod,leg1ResetFrequencyPeriod,leg2ResetFrequencyPeriod,optionExerciseEndDate,leg2FixedRate,upi,fisn,upiUnderlierName,notionalCapIndicator,otherPaymentCapIndicator
24,NEWT,NaN,IR,True,N,NaN,False,NaN,SDRIR91073389,2025-01-15T01:16:38Z,...,NaN,NaN,YEAR,NaN,NaN,QZ80KMBF5CF8,NA/Swap Infl Idx GBP,UK-RPI,N,NaN
25,NEWT,NaN,IR,True,N,NaN,False,NaN,SDRIR91073391,2025-01-15T01:16:38Z,...,NaN,NaN,YEAR,NaN,NaN,QZ80KMBF5CF8,NA/Swap Infl Idx GBP,UK-RPI,N,NaN
82,NEWT,NaN,IR,True,N,NaN,False,NaN,SDRIR91123577,2025-01-17T15:37:57Z,...,NaN,NaN,YEAR,NaN,NaN,QZ80KMBF5CF8,NA/Swap Infl Idx GBP,UK-RPI,N,NaN
83,NEWT,NaN,IR,True,N,NaN,False,NaN,SDRIR91123581,2025-01-17T15:37:57Z,...,NaN,NaN,YEAR,NaN,NaN,QZ80KMBF5CF8,NA/Swap Infl Idx GBP,UK-RPI,N,NaN
84,NEWT,NaN,IR,True,N,NaN,False,NaN,SDRIR91123575,2025-01-17T15:37:57Z,...,NaN,NaN,YEAR,NaN,NaN,QZ80KMBF5CF8,NA/Swap Infl Idx GBP,UK-RPI,N,NaN


In [3]:
## Check product types and missing values

# Check product classifications
print("UPI distribution:")
print(gbp_df["upi"].value_counts(dropna=False))

print("\nFISN distribution:")
print(gbp_df["fisn"].value_counts(dropna=False))

# Check important fields
required_cols = [
    "disseminationIdentifier",
    "executionDateTime",
    "leg1NotionalAmount",
    "leg1NotionalCurrency",
    "effectiveDate",
    "expirationDate"
]

print("\nMissing values:")
print(gbp_df[required_cols].isna().sum())

UPI distribution:
upi
QZ80KMBF5CF8    56
QZ0LNG9Z6KLB    34
QZ0W41WLZFFQ     8
QZWP6249QPX9     4
Name: count, dtype: int64

FISN distribution:
fisn
NA/Swap Infl Idx GBP    56
NA/Swap OIS GBP         46
Name: count, dtype: int64

Missing values:
disseminationIdentifier    0
executionDateTime          0
leg1NotionalAmount         0
leg1NotionalCurrency       0
effectiveDate              0
expirationDate             0
dtype: int64


In [4]:
## Extract GBP OIS transactions
# Filter GBP overnight index swaps
ois_df = gbp_df[
    gbp_df["fisn"].eq("NA/Swap OIS GBP")
].copy()

# Check unique identifiers
print("Total GBP OIS records:", len(ois_df))
print("Unique identifiers:",
      ois_df["disseminationIdentifier"].nunique())

# Check notional values and trade dates
display(ois_df[
    [
        "disseminationIdentifier",
        "executionDateTime",
        "leg1NotionalAmount",
        "leg1NotionalCurrency",
        "effectiveDate",
        "expirationDate"
    ]
].head())

Total GBP OIS records: 46
Unique identifiers: 28


,disseminationIdentifier,executionDateTime,leg1NotionalAmount,leg1NotionalCurrency,effectiveDate,expirationDate
176,SDRIR91238342,2025-01-27T10:44:11Z,50000000,GBP,2025-07-28,2027-07-28
178,SDRIR91238333,2025-01-27T10:44:11Z,12000000,GBP,2025-07-28,2035-07-28
187,SDRIR91237936,2025-01-27T10:38:35Z,600000,GBP,2025-07-28,2035-07-28
188,SDRIR91237927,2025-01-27T10:38:35Z,50000000,GBP,2025-07-28,2027-07-28
189,SDRIR91237922,2025-01-27T10:38:35Z,6000000,GBP,2025-07-28,2027-07-28


In [5]:
# Convert dates
ois_df["effectiveDate"] = pd.to_datetime(
    ois_df["effectiveDate"], errors="coerce"
)

ois_df["expirationDate"] = pd.to_datetime(
    ois_df["expirationDate"], errors="coerce"
)

# Calculate swap tenor in years
ois_df["tenor_years"] = (
    (ois_df["expirationDate"] - ois_df["effectiveDate"]).dt.days / 365.25
).round(2)

# Validate dates and tenor
print("Invalid tenors:",
      (ois_df["tenor_years"].isna() |
       (ois_df["tenor_years"] <= 0)).sum())

print("\nTenor distribution:")
print(ois_df["tenor_years"].value_counts().sort_index())



Invalid tenors: 0

Tenor distribution:
tenor_years
0.11      4
0.13      4
2.00     19
10.00    19
Name: count, dtype: int64


In [6]:
## Check null values
ois_df.isnull().count()


action                      46
amendmentIndicator          46
assetClass                  46
blockTradeIndicator         46
cleared                     46
                            ..
fisn                        46
upiUnderlierName            46
notionalCapIndicator        46
otherPaymentCapIndicator    46
tenor_years                 46
Length: 149, dtype: int64

In [7]:
# Columns where every value is missing (NaN/None)
empty_columns = ois_df.columns[ois_df.isna().all()]

print(f"Completely empty columns: {len(empty_columns)} out of {ois_df.shape[1]}")
print(empty_columns.tolist())

Completely empty columns: 107 out of 149
['amendmentIndicator', 'coupon', 'deliveryType', 'embeddedOptionType', 'exchangeRate', 'exchangeRateBasis', 'firstExerciseDate', 'futureValueNotional', 'indexFactor', 'instrumentType', 'leg1CallAmount', 'leg1CallCurrency', 'leg1CommodityDesc', 'leg1FloatingRatePaymentFrequencyMultiplier', 'leg1FloatingRatePaymentFrequencyPeriod', 'leg1NotionalEndDate', 'leg1NotionalQuantity', 'leg1NotionalScheduleType', 'leg1PhysicalDeliveryLocation', 'leg1PhysicalDeliveryRegion', 'leg1PricingIndexLocation', 'leg1PricingIndexRegion', 'leg1PutAmount', 'leg1PutCurrency', 'leg1QuantityFrequencyMultiplier', 'leg1QuantityUnit', 'leg1ResetFrequencyMultiplier', 'leg1SettlementLocation', 'leg1Spread', 'leg1SpreadCurrency', 'leg1SpreadNotation', 'leg1TotalNotionalQuantity', 'leg1UnderlierCurrency', 'leg1UnderlierID', 'leg1UnderlierIDSource', 'leg1UnderlierTenorMultiplier', 'leg1UnderlierTenorPeriod', 'leg1UnderlyingAssetOrContractSubType', 'leg1UnderlyingAssetOrContractT

In [8]:
## Drop the empty columns
selected_columns = [
    "disseminationIdentifier",
    "executionDateTime",
    "effectiveDate",
    "expirationDate",
    "leg1NotionalAmount",
    "leg1NotionalCurrency",
    "tenor_years",
    "upi",
    "fisn",
    "upiUnderlierName",
    "action",
    "event"
]

cme_clean = ois_df[selected_columns].copy()

print("Rows:", len(cme_clean))
print("Columns:", len(cme_clean.columns))
print("Missing values:", cme_clean.isna().sum().sum())

display(cme_clean.head())

Rows: 46
Columns: 12
Missing values: 0


,disseminationIdentifier,executionDateTime,effectiveDate,expirationDate,leg1NotionalAmount,leg1NotionalCurrency,tenor_years,upi,fisn,upiUnderlierName,action,event
176,SDRIR91238342,2025-01-27T10:44:11Z,2025-07-28,2027-07-28,50000000,GBP,2.0,QZ0LNG9Z6KLB,NA/Swap OIS GBP,GBP-SONIA-OIS Compound,NEWT,TRAD
178,SDRIR91238333,2025-01-27T10:44:11Z,2025-07-28,2035-07-28,12000000,GBP,10.0,QZ0LNG9Z6KLB,NA/Swap OIS GBP,GBP-SONIA-OIS Compound,NEWT,TRAD
187,SDRIR91237936,2025-01-27T10:38:35Z,2025-07-28,2035-07-28,600000,GBP,10.0,QZ0LNG9Z6KLB,NA/Swap OIS GBP,GBP-SONIA-OIS Compound,NEWT,TRAD
188,SDRIR91237927,2025-01-27T10:38:35Z,2025-07-28,2027-07-28,50000000,GBP,2.0,QZWP6249QPX9,NA/Swap OIS GBP,GBP-SONIA-OIS Compound,NEWT,TRAD
189,SDRIR91237922,2025-01-27T10:38:35Z,2025-07-28,2027-07-28,6000000,GBP,2.0,QZ0LNG9Z6KLB,NA/Swap OIS GBP,GBP-SONIA-OIS Compound,NEWT,TRAD


In [9]:
# Rename CME fields to NorthBridge schema
real_otc_df = cme_clean.rename(columns={
    "disseminationIdentifier": "trade_id",
    "executionDateTime": "trade_timestamp",
    "leg1NotionalAmount": "notional_amount",
    "leg1NotionalCurrency": "currency",
    "fisn": "product_type",
    "effectiveDate": "effective_date",
    "expirationDate": "expiration_date"
}).copy()

# Standardise the verified GBP OIS classification
real_otc_df["product_type"] = "GBP_OIS"

# Keep compatible fields and useful source metadata
selected_columns = [
    "trade_id",
    "trade_timestamp",
    "product_type",
    "currency",
    "notional_amount",
    "tenor_years",
    "effective_date",
    "expiration_date",
    "upi",
    "upiUnderlierName",
    "action",
    "event"
]

real_otc_df = real_otc_df[selected_columns]

print("Records:", len(real_otc_df))
print("Columns:", len(real_otc_df.columns))

display(real_otc_df.head())

Records: 46
Columns: 12


,trade_id,trade_timestamp,product_type,currency,notional_amount,tenor_years,effective_date,expiration_date,upi,upiUnderlierName,action,event
176,SDRIR91238342,2025-01-27T10:44:11Z,GBP_OIS,GBP,50000000,2.0,2025-07-28,2027-07-28,QZ0LNG9Z6KLB,GBP-SONIA-OIS Compound,NEWT,TRAD
178,SDRIR91238333,2025-01-27T10:44:11Z,GBP_OIS,GBP,12000000,10.0,2025-07-28,2035-07-28,QZ0LNG9Z6KLB,GBP-SONIA-OIS Compound,NEWT,TRAD
187,SDRIR91237936,2025-01-27T10:38:35Z,GBP_OIS,GBP,600000,10.0,2025-07-28,2035-07-28,QZ0LNG9Z6KLB,GBP-SONIA-OIS Compound,NEWT,TRAD
188,SDRIR91237927,2025-01-27T10:38:35Z,GBP_OIS,GBP,50000000,2.0,2025-07-28,2027-07-28,QZWP6249QPX9,GBP-SONIA-OIS Compound,NEWT,TRAD
189,SDRIR91237922,2025-01-27T10:38:35Z,GBP_OIS,GBP,6000000,2.0,2025-07-28,2027-07-28,QZ0LNG9Z6KLB,GBP-SONIA-OIS Compound,NEWT,TRAD


In [10]:
print(gbp_df["fisn"].value_counts(dropna=False))
print("Actual OIS records:",
      gbp_df["fisn"].eq("NA/Swap OIS GBP").sum())

fisn
NA/Swap Infl Idx GBP    56
NA/Swap OIS GBP         46
Name: count, dtype: int64
Actual OIS records: 46


In [11]:
print("Cleaned records:", len(real_otc_df))
print("Product types:")
print(real_otc_df["product_type"].value_counts())

print("Duplicate trade IDs:",
      real_otc_df["trade_id"].duplicated().sum())

Cleaned records: 46
Product types:
product_type
GBP_OIS    46
Name: count, dtype: int64
Duplicate trade IDs: 18


In [12]:
## Check duplicate trades
duplicate_trades = real_otc_df[
    real_otc_df.duplicated("trade_id", keep=False)
].sort_values("trade_id")

display(duplicate_trades[
    [
        "trade_id",
        "trade_timestamp",
        "notional_amount",
        "tenor_years",
        "action",
        "event"
    ]
].head(20))

,trade_id,trade_timestamp,notional_amount,tenor_years,action,event
385,SDRIR90975742,2025-01-06T10:40:21Z,1800000000,0.11,NEWT,TRAD
284,SDRIR90975742,2025-01-06T10:40:21Z,1800000000,0.11,NEWT,TRAD
384,SDRIR90975743,2025-01-06T10:40:21Z,1600000000,0.13,NEWT,TRAD
283,SDRIR90975743,2025-01-06T10:40:21Z,1600000000,0.13,NEWT,TRAD
404,SDRIR91052317,2025-01-13T09:51:02Z,700000000,0.11,NEWT,TRAD
761,SDRIR91052317,2025-01-13T09:51:02Z,700000000,0.11,NEWT,TRAD
760,SDRIR91052319,2025-01-13T09:51:02Z,600000000,0.13,NEWT,TRAD
403,SDRIR91052319,2025-01-13T09:51:02Z,600000000,0.13,NEWT,TRAD
635,SDRIR91237922,2025-01-27T10:38:35Z,6000000,2.00,NEWT,TRAD
189,SDRIR91237922,2025-01-27T10:38:35Z,6000000,2.00,NEWT,TRAD


In [ ]:
## Remove exact duplicates only
display= real_otc_df.drop_duplicates().copy()

print("Records after cleaning:", len(real_otc_df))
print("Remaining duplicate IDs:",
      real_otc_df.duplicated("trade_id").sum())

Records after cleaning: 28
Remaining duplicate IDs: 0


In [21]:
real_otc_df.head()

,trade_id,trade_timestamp,product_type,currency,notional_amount,tenor_years,effective_date,expiration_date,upi,upiUnderlierName,action,event
176,SDRIR91238342,2025-01-27T10:44:11Z,GBP_OIS,GBP,50000000,2.0,2025-07-28,2027-07-28,QZ0LNG9Z6KLB,GBP-SONIA-OIS Compound,NEWT,TRAD
178,SDRIR91238333,2025-01-27T10:44:11Z,GBP_OIS,GBP,12000000,10.0,2025-07-28,2035-07-28,QZ0LNG9Z6KLB,GBP-SONIA-OIS Compound,NEWT,TRAD
187,SDRIR91237936,2025-01-27T10:38:35Z,GBP_OIS,GBP,600000,10.0,2025-07-28,2035-07-28,QZ0LNG9Z6KLB,GBP-SONIA-OIS Compound,NEWT,TRAD
188,SDRIR91237927,2025-01-27T10:38:35Z,GBP_OIS,GBP,50000000,2.0,2025-07-28,2027-07-28,QZWP6249QPX9,GBP-SONIA-OIS Compound,NEWT,TRAD
189,SDRIR91237922,2025-01-27T10:38:35Z,GBP_OIS,GBP,6000000,2.0,2025-07-28,2027-07-28,QZ0LNG9Z6KLB,GBP-SONIA-OIS Compound,NEWT,TRAD


In [14]:
## Save the cleaned dataset

'''
real_otc_df.to_csv(
    "data/CME_Swap_data/cme_gbp_swaps_clean.csv",
    index=False
)

print("Final dataset saved:", real_otc_df.shape)

 '''

Final dataset saved: (28, 12)
